In [ ]:
from pathlib import Path
import csv
import html
import json
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform as transform_coordinates
from pyproj import Geod
from IPython.display import display, HTML

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.geofm-root').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the project containing .geofm-root.')

# Change this path if the dataset is moved; no data are copied or modified.
DATASET_DIR = Path(r'D:/UNI/GeoTest/data/sen1floods11/HandLabeled')
CATALOG_DIR = DATASET_DIR / 'catalog' / 'sen1floods11_hand_labeled_source'
RASTER_DIR = DATASET_DIR / 'HandLabeled'
SPLIT_DIR = DATASET_DIR / 'flood_handlabeled'
OUTPUT_DIR = ROOT / 'reports' / 'sen1floods11_geographic_coverage'
READ_RASTER_METADATA = True
assert CATALOG_DIR.is_dir(), f'Catalog not found: {CATALOG_DIR}'
geod = Geod(ellps='WGS84')
print('Catalog:', CATALOG_DIR)
print('Footprints describe dataset chips, not full Sentinel acquisition scenes.')
print('The interactive basemap requires internet; no rasters are uploaded.')

In [ ]:
split_files = {
    'train': 'flood_train_data.csv',
    'validation': 'flood_valid_data.csv',
    'test': 'flood_test_data.csv',
    'bolivia_test': 'flood_bolivia_data.csv',
}
split_by_id = {}
split_audit = []
for split, filename in split_files.items():
    path = SPLIT_DIR / filename
    with path.open(newline='', encoding='utf-8-sig') as stream:
        entries = [row for row in csv.reader(stream) if row]
    ids = [Path(row[0].replace('\\', '/')).name.removesuffix('_S1Hand.tif') for row in entries]
    if len(ids) != len(set(ids)):
        raise ValueError(f'Duplicate sample in {filename}')
    for chip_id in ids:
        if chip_id in split_by_id:
            raise ValueError(f'Split overlap: {chip_id}')
        split_by_id[chip_id] = split
    split_audit.append({'split': split, 'samples': len(ids), 'csv': str(path),
                        'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
display(pd.DataFrame(split_audit)[['split', 'samples', 'csv']])

In [ ]:
def footprint_measurements(geometry):
    # Catalog items are WGS84 polygons. Measure on the ellipsoid, not degrees.
    if geometry['type'] != 'Polygon':
        raise ValueError(f"Unexpected geometry: {geometry['type']}")
    rings = geometry['coordinates']
    areas = [abs(geod.polygon_area_perimeter(*zip(*ring))[0]) for ring in rings]
    ring = rings[0]
    west, east = min(p[0] for p in ring), max(p[0] for p in ring)
    south, north = min(p[1] for p in ring), max(p[1] for p in ring)
    lon, lat = (west + east) / 2, (south + north) / 2
    return {'longitude': lon, 'latitude': lat,
            'width_km': geod.inv(west, lat, east, lat)[2] / 1000,
            'height_km': geod.inv(lon, south, lon, north)[2] / 1000,
            'area_km2': (areas[0] - sum(areas[1:])) / 1e6}

rows, features = [], []
for path in sorted(CATALOG_DIR.glob('*/*.json')):
    item = json.loads(path.read_text(encoding='utf-8'))
    if item.get('type') != 'Feature':
        continue
    chip_id = item['id']
    props = item['properties']
    row = {'chip_id': chip_id, 'event_region': props.get('country', chip_id.rsplit('_', 1)[0]),
           'split': split_by_id.get(chip_id, 'unassigned'),
           'catalog_datetime': props.get('datetime'), 'catalog_file': str(path),
           **footprint_measurements(item['geometry'])}
    # Only read raster headers. S2 dates are not inferred from the S1 catalog date.
    for sensor in ['S1', 'S2']:
        raster_path = RASTER_DIR / f'{sensor}Hand' / f'{chip_id}_{sensor}Hand.tif'
        row[f'{sensor}_available'] = raster_path.is_file()
        if READ_RASTER_METADATA and raster_path.is_file():
            with rasterio.open(raster_path) as ds:
                row.update({f'{sensor}_width_px': ds.width, f'{sensor}_height_px': ds.height,
                            f'{sensor}_bands': ds.count, f'{sensor}_crs': str(ds.crs),
                            f'{sensor}_file_MiB': raster_path.stat().st_size / 2**20})
                if ds.crs is None:
                    raise ValueError(f'Raster CRS missing: {raster_path}')
                corners = [ds.transform * xy for xy in [(0, 0), (ds.width, 0),
                           (ds.width, ds.height), (0, ds.height), (0, 0)]]
                xs, ys = transform_coordinates(ds.crs, 'EPSG:4326', *zip(*corners))
                raster_geometry = {'type': 'Polygon', 'coordinates': [list(map(list, zip(xs, ys)))]}
                measures = footprint_measurements(raster_geometry)
                row[f'{sensor}_area_km2'] = measures['area_km2']
                row[f'{sensor}_pixel_width_m'] = measures['width_km'] * 1000 / ds.width
                row[f'{sensor}_pixel_height_m'] = measures['height_km'] * 1000 / ds.height
                row[f'{sensor}_catalog_bbox_max_difference_deg'] = max(abs(a-b) for a,b in zip(
                    [min(xs), min(ys), max(xs), max(ys)], item['bbox']))
    rows.append(row)
    features.append({'type': 'Feature', 'geometry': item['geometry'], 'properties': row})
chips = pd.DataFrame(rows)
if chips.empty or chips.chip_id.duplicated().any():
    raise ValueError('Empty catalog or duplicate chip IDs.')
missing = set(split_by_id) - set(chips.chip_id)
if missing:
    raise ValueError(f'Split samples missing from catalog: {sorted(missing)}')
collection = {'type': 'FeatureCollection', 'features': features}
print(f'{len(chips)} unique chips across {chips.event_region.nunique()} event regions.')
print('All catalog samples are retained, including ignored-label or missing-raster samples.')
print('Catalog datetime is source metadata, not independently verified S2 acquisition time.')
display(chips.head())

In [ ]:
event_summary = chips.groupby('event_region').agg(
    chips=('chip_id', 'size'), summed_chip_area_km2=('area_km2', 'sum'),
    min_chip_area_km2=('area_km2', 'min'), max_chip_area_km2=('area_km2', 'max'),
    first_catalog_date=('catalog_datetime', 'min'), last_catalog_date=('catalog_datetime', 'max'),
).sort_values('chips', ascending=False)
display(event_summary)
display(pd.crosstab(chips.event_region, chips.split))
display(chips[['width_km', 'height_km', 'area_km2']].describe())
print('Summed chip area counts overlaps; it is not the union area or the flooded area.')
print('Region names are catalog event labels (e.g. Mekong), not a country/biome classification.')
if READ_RASTER_METADATA:
    cols = [c for c in chips if 'bbox_max_difference' in c]
    print('Maximum catalog/raster bounding-box differences in degrees:')
    display(chips[cols].max())
    display(chips[['S1_available', 'S2_available']].value_counts().rename('chips'))

In [ ]:
# Leaflet provides the world map without adding Python mapping dependencies.
# Circle markers show locations at world scale; polygons show actual footprint sizes when zoomed in.
map_template = r"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>Sen1Floods11 hand-labelled coverage</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>body{margin:0;font:14px Arial}#toolbar{padding:12px;background:#f4f6f8}
#map{height:690px}.legend{background:white;padding:12px;line-height:1.7}
select,button{margin:4px;padding:5px}</style></head><body>
<div id="toolbar"><b>Sen1Floods11: hand-labelled chip footprints</b><br>
Region <select id="region"><option value="all">All regions</option></select>
Split <select id="split"><option value="all">All splits</option></select>
<button id="fit">Fit selection</button><button id="world">World view</button>
<span id="count"></span><br>Circles are location indicators, not acquisition sizes.
Zoom in to see chip boundaries. Areas are ground footprint areas, not water areas.</div>
<div id="map"></div><script>
const data=__GEOJSON__;
const colors={train:'#0072B2',validation:'#E69F00',test:'#009E73',bolivia_test:'#CC79A7',unassigned:'#777777'};
const map=L.map('map').setView([15,5],2);
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',{
maxZoom:19,attribution:'&copy; OpenStreetMap contributors'}).addTo(map);
L.control.scale({imperial:false}).addTo(map);
const footprints=L.featureGroup().addTo(map), markers=L.featureGroup().addTo(map);
L.control.layers(null,{'Chip footprints':footprints,'Location markers':markers}).addTo(map);
function esc(v){const e=document.createElement('span');e.textContent=String(v??'Unavailable');return e.innerHTML;}
function fmt(v,n=2){return typeof v==='number'?v.toFixed(n):'Unavailable';}
for(const [key,field] of [['region','event_region'],['split','split']]){
 for(const value of [...new Set(data.features.map(f=>f.properties[field]))].sort()){
 const o=document.createElement('option');o.value=value;o.textContent=value;document.getElementById(key).appendChild(o);}}
function render(){footprints.clearLayers();markers.clearLayers();let count=0;
 for(const f of data.features){const p=f.properties;
 if(document.getElementById('region').value!=='all'&&p.event_region!==document.getElementById('region').value)continue;
 if(document.getElementById('split').value!=='all'&&p.split!==document.getElementById('split').value)continue;
 count++;const color=colors[p.split]||'#777';
 let popup=`<b>${esc(p.chip_id)}</b><br>Event region: ${esc(p.event_region)}<br>Split: ${esc(p.split)}
 <br>Catalog source date: ${esc(p.catalog_datetime)}<br>Centre: ${fmt(p.latitude,5)}, ${fmt(p.longitude,5)}
 <br>Width / height: ${fmt(p.width_km)} / ${fmt(p.height_km)} km<br>Area: ${fmt(p.area_km2)} km?`;
 for(const s of ['S1','S2']){popup+=`<br><b>${s}</b>: local raster ${p[s+'_available']?'present':'missing'}
 <br>Pixels: ${esc(p[s+'_width_px'])} ? ${esc(p[s+'_height_px'])}; bands: ${esc(p[s+'_bands'])}
 <br>Approx. ground pixel: ${fmt(p[s+'_pixel_width_m'])} ? ${fmt(p[s+'_pixel_height_m'])} m
 <br>CRS: ${esc(p[s+'_crs'])}; file: ${fmt(p[s+'_file_MiB'])} MiB`;}
 L.geoJSON(f,{style:{color,weight:2,fillOpacity:0.15}}).bindPopup(popup).addTo(footprints);
 L.circleMarker([p.latitude,p.longitude],{radius:4,color,weight:1,fillOpacity:0.8})
 .bindTooltip(p.chip_id+' | '+p.split).bindPopup(popup).addTo(markers);}
 document.getElementById('count').textContent=count+' chips';}
function fit(){if(footprints.getLayers().length)map.fitBounds(footprints.getBounds(),{padding:[25,25],maxZoom:12});}
for(const id of ['region','split'])document.getElementById(id).onchange=()=>{render();fit();};
document.getElementById('fit').onclick=fit;
document.getElementById('world').onclick=()=>map.setView([15,5],2);
const legend=L.control({position:'bottomleft'});legend.onAdd=function(){const d=L.DomUtil.create('div','legend');
 d.innerHTML=Object.entries(colors).map(([k,c])=>`<span style="color:${c}">?</span> ${k}`).join('<br>');return d;};legend.addTo(map);
render();</script></body></html>"""
map_html = map_template.replace('__GEOJSON__', json.dumps(collection, allow_nan=False).replace('</', r'<\/'))
display(HTML('<iframe style="width:100%;height:810px;border:0" srcdoc="' + html.escape(map_html, quote=True) + '"></iframe>'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pd.crosstab(chips.event_region, chips.split).plot.bar(stacked=True, ax=axes[0])
axes[0].set(title='Hand-labelled samples by event region and split', ylabel='Number of chips', xlabel='Event region')
chips.boxplot(column='area_km2', by='event_region', ax=axes[1], rot=65)
axes[1].set(title='Chip footprint area by event region', ylabel='Ground area (km?)', xlabel='Event region')
fig.suptitle('')
fig.tight_layout()
plt.show()
# EPSG:4326 chips have angular pixel spacing, so their physical width varies with latitude.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'world_map.html').write_text(map_html, encoding='utf-8')
(OUTPUT_DIR / 'chip_footprints.geojson').write_text(json.dumps(collection, indent=2, allow_nan=False), encoding='utf-8')
chips.to_csv(OUTPUT_DIR / 'chip_inventory.csv', index=False)
event_summary.to_csv(OUTPUT_DIR / 'event_summary.csv')
pd.DataFrame(split_audit).to_csv(OUTPUT_DIR / 'split_audit.csv', index=False)
fig.savefig(OUTPUT_DIR / 'coverage_summary.png', dpi=180, bbox_inches='tight')
print('Saved map, footprints, tables and summary figure to:', OUTPUT_DIR)
print('Open world_map.html in a browser if the notebook renderer blocks the interactive map.')